In [10]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import numpy as np
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
from torch import nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

## 1) Load data


In [11]:
anime = pd.read_csv("anime_data.csv")
creators = pd.read_csv("creators.csv")
genres = pd.read_csv("genres.csv")
subs = pd.read_csv("subs_eng.csv")  # for Task2 mostly; not needed for Task1 directly

# Ensure consistent node ordering
anime = anime.sort_values("anime_id").reset_index(drop=True)
anime_ids = anime["anime_id"].values
id2idx = {aid:i for i, aid in enumerate(anime_ids)}

# Target
y = anime["num_subs_eng"].fillna(0).astype(float).values  # usually not missing, but safe

## 2) Build bipartite incidence matrix B and project to anime-anime with Jaccard

In [12]:
def build_B(anime_ids, links_df, item_col):
    """
    links_df has columns: anime_id, item_col (creator_id or genre_id)
    returns sparse B: shape (n_anime, n_items)
    """
    df = links_df[links_df["anime_id"].isin(anime_ids)].copy()
    df["a_idx"] = df["anime_id"].map(id2idx)
    # map items to 0..m-1
    items = df[item_col].astype(str).unique()
    item2j = {it:j for j, it in enumerate(items)}
    df["i_idx"] = df[item_col].astype(str).map(item2j)

    rows = df["a_idx"].values
    cols = df["i_idx"].values
    data = np.ones(len(df), dtype=np.float32)
    B = sp.csr_matrix((data, (rows, cols)), shape=(len(anime_ids), len(items)))
    return B

def jaccard_projection(B, threshold=0.0, topk=None):
    """
    Compute anime-anime Jaccard weights from incidence matrix B.
    intersection = B @ B.T  (counts of shared items)
    union = deg[i] + deg[j] - intersection
    weight = intersection / union
    - threshold: drop edges below weight
    - topk: optional keep only topk neighbors per node (by weight)
    Returns edge_index (2,E), edge_weight (E,)
    """
    # intersection counts
    inter = (B @ B.T).tocsr()
    inter.setdiag(0)
    inter.eliminate_zeros()

    deg = np.array(B.sum(axis=1)).flatten().astype(np.float32)
    # compute weights on nonzero entries only
    inter_coo = inter.tocoo()
    i = inter_coo.row
    j = inter_coo.col
    c = inter_coo.data.astype(np.float32)

    union = deg[i] + deg[j] - c
    w = c / np.maximum(union, 1e-8)

    # threshold filter
    if threshold > 0:
        mask = w >= threshold
        i, j, w = i[mask], j[mask], w[mask]

    # optionally keep topk per i
    if topk is not None:
        # group by source node i and keep topk weights
        order = np.lexsort((-w, i))   # sort by i asc, w desc
        i2, j2, w2 = i[order], j[order], w[order]
        keep = np.zeros_like(w2, dtype=bool)
        # mark first topk entries per i
        last = -1
        cnt = 0
        for idx, src in enumerate(i2):
            if src != last:
                last = src
                cnt = 0
            if cnt < topk:
                keep[idx] = True
                cnt += 1
        i, j, w = i2[keep], j2[keep], w2[keep]

    # make undirected (optional but recommended)
    i_all = np.concatenate([i, j])
    j_all = np.concatenate([j, i])
    w_all = np.concatenate([w, w])

    edge_index = torch.tensor(np.vstack([i_all, j_all]), dtype=torch.long)
    edge_weight = torch.tensor(w_all, dtype=torch.float32)
    return edge_index, edge_weight

## Choose network option


In [13]:
# Option A: creators
B_cre = build_B(anime_ids, creators, "creator_id")
edge_index_A, edge_weight_A = jaccard_projection(B_cre, threshold=0.05, topk=50)

# Option B: genres
B_gen = build_B(anime_ids, genres, "genre_id")
edge_index_B, edge_weight_B = jaccard_projection(B_gen, threshold=0.05, topk=50)

# Option C: combined (stack creators+genres as distinct tokens)
creators_tmp = creators.copy()
creators_tmp["joint_item"] = "c_" + creators_tmp["creator_id"].astype(str)
genres_tmp = genres.copy()
genres_tmp["joint_item"] = "g_" + genres_tmp["genre_id"].astype(str)
joint = pd.concat([
    creators_tmp[["anime_id", "joint_item"]].rename(columns={"joint_item":"item"}),
    genres_tmp[["anime_id", "joint_item"]].rename(columns={"joint_item":"item"})
], ignore_index=True)
B_joint = build_B(anime_ids, joint.rename(columns={"item":"item_col"}), "item_col")
edge_index_C, edge_weight_C = jaccard_projection(B_joint, threshold=0.05, topk=50)

# pick one for now
edge_index, edge_weight = edge_index_C, edge_weight_C

# 3) Build node features X


In [14]:
feature_cols = ["top_home_sales", "awards", "num_creators", "num_genres", "aired_year"]
X_df = anime[feature_cols].copy()

# fill missing safely
for c in feature_cols:
    X_df[c] = X_df[c].fillna(X_df[c].median())

X = X_df.values.astype(np.float32)

# Train/val/test split (node-level masks)
n = len(anime)
idx = np.arange(n)

idx_train, idx_tmp = train_test_split(idx, test_size=0.4, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=42)

train_mask = torch.zeros(n, dtype=torch.bool); train_mask[idx_train] = True
val_mask   = torch.zeros(n, dtype=torch.bool); val_mask[idx_val] = True
test_mask  = torch.zeros(n, dtype=torch.bool); test_mask[idx_test] = True

# Standardize using TRAIN ONLY (avoid leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X[idx_train])
X_scaled = X.copy()
X_scaled[idx_train] = X_train_scaled
X_scaled[idx_val] = scaler.transform(X[idx_val])
X_scaled[idx_test] = scaler.transform(X[idx_test])

# PyG Data object
data = Data(
    x=torch.tensor(X_scaled, dtype=torch.float32),
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=torch.tensor(y, dtype=torch.float32),
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
)

## 4) GNN model (start simple, then we’ll tune)


In [15]:
class GNNRegressor(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=1, dropout=0.3, layer="gcn"):
        super().__init__()
        self.dropout = dropout
        if layer == "gcn":
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, hidden)
        elif layer == "sage":
            self.conv1 = SAGEConv(in_dim, hidden)
            self.conv2 = SAGEConv(hidden, hidden)
        elif layer == "gat":
            # GAT ignores edge_weight; you can still use unweighted edges
            self.conv1 = GATConv(in_dim, hidden, heads=2, concat=False)
            self.conv2 = GATConv(hidden, hidden, heads=2, concat=False)
        else:
            raise ValueError("layer must be gcn/sage/gat")

        self.lin1 = nn.Linear(hidden, hidden)
        self.lin2 = nn.Linear(hidden, out_dim)

    def forward(self, data):
        x, ei = data.x, data.edge_index

        if isinstance(self.conv1, GCNConv):
            ew = data.edge_weight
            x = self.conv1(x, ei, ew)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.conv2(x, ei, ew)
        else:
            x = self.conv1(x, ei)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.conv2(x, ei)

        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        out = self.lin2(x).squeeze(-1)
        return out

def eval_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

## 5) Train loop with early stopping on val MAE


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

model = GNNRegressor(in_dim=data.x.size(1), hidden=64, dropout=0.3, layer="gcn").to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

best_val = float("inf")
best_state = None
patience, patience_left = 30, 30

for epoch in range(1, 401):
    model.train()
    opt.zero_grad()
    pred = model(data)
    loss = F.l1_loss(pred[data.train_mask], data.y[data.train_mask])  # MAE loss
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(data).detach().cpu().numpy()
        y_val = data.y[data.val_mask].cpu().numpy()
        p_val = pred[data.val_mask.cpu().numpy()]
        val_mae, _, _ = eval_metrics(y_val, p_val)

    if val_mae < best_val - 1e-4:
        best_val = val_mae
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1
        if patience_left == 0:
            break

# load best
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    pred = model(data).cpu().numpy()

y_test = data.y[data.test_mask].cpu().numpy()
p_test = pred[data.test_mask.cpu().numpy()]
mae, rmse, r2 = eval_metrics(y_test, p_test)
print("GNN Test:", {"MAE": mae, "RMSE": rmse, "R2": r2})

GNN Test: {'MAE': 6.472672939300537, 'RMSE': np.float64(10.548149517899011), 'R2': 0.439596951007843}


## 6) Baseline: Gradient boosting (fair: same X, same split)


In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor

gb = HistGradientBoostingRegressor(random_state=42)
gb.fit(X_scaled[idx_train], y[idx_train])

p_test_gb = gb.predict(X_scaled[idx_test])
mae_gb, rmse_gb, r2_gb = eval_metrics(y[idx_test], p_test_gb)
print("GB Test:", {"MAE": mae_gb, "RMSE": rmse_gb, "R2": r2_gb})

# permutation importance for baseline
from sklearn.inspection import permutation_importance
pi = permutation_importance(gb, X_scaled[idx_test], y[idx_test], n_repeats=20, random_state=42)
imp = pd.Series(pi.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Permutation importance:\n", imp)


GB Test: {'MAE': 6.378186455428203, 'RMSE': np.float64(9.504852163765714), 'R2': 0.544971397227366}
Permutation importance:
 num_genres        0.504284
awards            0.221723
aired_year        0.064930
top_home_sales    0.046478
num_creators      0.036005
dtype: float64


In [3]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00


# Part Two

In [20]:
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE

# If available
from gensim.models import Word2Vec

# -------------------------
# 1) Load data
# -------------------------
anime = pd.read_csv("anime_data.csv")
creators = pd.read_csv("creators.csv")
genres = pd.read_csv("genres.csv")
subs = pd.read_csv("subs_eng.csv")

anime_ids = anime["anime_id"].unique()
anime_set = set(anime_ids)

## 2) Build bipartite adjacency lists We'll prefix node ids so anime/genre/creator don't collide


In [21]:
def build_bipartite_adj(df, left_col, right_col, left_prefix, right_prefix, restrict_left=None):
    adj = defaultdict(list)
    df = df.copy()
    if restrict_left is not None:
        df = df[df[left_col].isin(restrict_left)]
    for l, r in zip(df[left_col].values, df[right_col].values):
        lnode = f"{left_prefix}{l}"
        rnode = f"{right_prefix}{r}"
        adj[lnode].append(rnode)
        adj[rnode].append(lnode)
    return adj

adj_gen = build_bipartite_adj(genres, "anime_id", "genre_id", "a_", "g_", restrict_left=anime_set)
adj_cre = build_bipartite_adj(creators, "anime_id", "creator_id", "a_", "c_", restrict_left=anime_set)

# 3) DeepWalk random walks + Word2Vec embeddings

In [22]:
rng = np.random.default_rng(42)

def random_walk(adj, start, walk_len=40):
    walk = [start]
    cur = start
    for _ in range(walk_len - 1):
        neigh = adj.get(cur, [])
        if not neigh:
            break
        cur = neigh[rng.integers(0, len(neigh))]
        walk.append(cur)
    return walk

def deepwalk_embeddings(adj, embed_dim=64, n_walks_per_node=10, walk_len=40, window=5, epochs=5, min_count=1, workers=4):
    nodes = list(adj.keys())
    walks = []
    for node in nodes:
        for _ in range(n_walks_per_node):
            walks.append(random_walk(adj, node, walk_len=walk_len))
    # train Word2Vec on walks
    w2v = Word2Vec(
        sentences=walks,
        vector_size=embed_dim,
        window=window,
        min_count=min_count,
        sg=1,          # skip-gram
        workers=workers,
        epochs=epochs
    )
    return w2v

# Train embeddings on each bipartite graph
w2v_gen = deepwalk_embeddings(adj_gen, embed_dim=64, n_walks_per_node=10, walk_len=40)
w2v_cre = deepwalk_embeddings(adj_cre, embed_dim=64, n_walks_per_node=10, walk_len=40)

def get_anime_emb(w2v, anime_id, dim):
    key = f"a_{anime_id}"
    if key in w2v.wv:
        return w2v.wv[key]
    return np.zeros(dim, dtype=np.float32)

E_gen = {aid: get_anime_emb(w2v_gen, aid, 64) for aid in anime_ids}
E_cre = {aid: get_anime_emb(w2v_cre, aid, 64) for aid in anime_ids}

# Stacked anime embeddings: 128d
E_anime = {aid: np.concatenate([E_gen[aid], E_cre[aid]]).astype(np.float32) for aid in anime_ids}

# Normalize for cosine scoring
anime_mat = np.vstack([E_anime[aid] for aid in anime_ids])
anime_mat = normalize(anime_mat)
aid2row = {aid:i for i, aid in enumerate(anime_ids)}

## 4) Temporal split (edges)


In [23]:
# Keep only rows for anime in anime_data
subs = subs[subs["anime_id"].isin(anime_set)].copy()

# choose split year = ~70% train edges (you can tune after checking distribution)
# quick heuristic: use quantile
split_year = int(subs["sub_year"].quantile(0.7))
train_edges = subs[subs["sub_year"] <= split_year][["anime_id", "group_id"]].drop_duplicates()
test_edges  = subs[subs["sub_year"] >  split_year][["anime_id", "group_id"]].drop_duplicates()

print("Split year:", split_year, "| train edges:", len(train_edges), "| test edges:", len(test_edges))

# All groups in test must have some history in train for meaningful recommendation
train_groups = set(train_edges["group_id"].unique())
test_edges = test_edges[test_edges["group_id"].isin(train_groups)]
print("Filtered test edges:", len(test_edges))

Split year: 2016 | train edges: 17124 | test edges: 6263
Filtered test edges: 2193


## 5) Group features from TRAIN only (mean pooling)

In [24]:
group2animes_train = defaultdict(list)
for a, g in zip(train_edges["anime_id"].values, train_edges["group_id"].values):
    group2animes_train[g].append(a)

def group_embedding(gid):
    animes = group2animes_train.get(gid, [])
    if not animes:
        return np.zeros(128, dtype=np.float32)
    M = np.vstack([E_anime[a] for a in animes])
    return M.mean(axis=0).astype(np.float32)

groups = sorted(list(train_groups))
E_group = {g: group_embedding(g) for g in groups}
group_mat = np.vstack([E_group[g] for g in groups])
group_mat = normalize(group_mat)
g2row = {g:i for i, g in enumerate(groups)}


## 6) Build positives/negatives for AUC evaluation

In [25]:
pos_test = list(zip(test_edges["anime_id"].values, test_edges["group_id"].values))
pos_set_all = set(zip(subs["anime_id"].values, subs["group_id"].values))  # all observed links

def sample_negatives(n_samples):
    neg = []
    while len(neg) < n_samples:
        a = anime_ids[rng.integers(0, len(anime_ids))]
        g = groups[rng.integers(0, len(groups))]
        if (a, g) not in pos_set_all:
            neg.append((a, g))
    return neg

neg_test = sample_negatives(len(pos_test))  # 1:1 ratio

# Scoring function: cosine between anime embedding and group taste embedding
def score_pair(a, g):
    if a not in aid2row or g not in g2row:
        return 0.0
    va = anime_mat[aid2row[a]]
    vg = group_mat[g2row[g]]
    return float(np.dot(va, vg))  # cosine since normalized

y_true = np.array([1]*len(pos_test) + [0]*len(neg_test))
y_score = np.array([score_pair(a,g) for (a,g) in pos_test] + [score_pair(a,g) for (a,g) in neg_test])

auc = roc_auc_score(y_true, y_score)
print("AUC-ROC:", auc)

AUC-ROC: 0.807473474548729


## 7) Precision@K and Recall@K (per group)


In [26]:
# For each group, recommend top-K anime not already in TRAIN history
train_history = {g: set(group2animes_train[g]) for g in groups}

# ground-truth future animes per group from TEST
gt_future = defaultdict(set)
for a,g in pos_test:
    gt_future[g].add(a)

def precision_recall_at_k(g, K=10):
    if g not in gt_future or len(gt_future[g]) == 0:
        return None

    # candidate anime = all anime excluding those already in train history
    candidates = [a for a in anime_ids if a not in train_history[g]]
    # score all candidates
    scores = np.array([score_pair(a,g) for a in candidates])
    topk_idx = np.argsort(-scores)[:K]
    recs = [candidates[i] for i in topk_idx]

    hits = sum([1 for a in recs if a in gt_future[g]])
    prec = hits / K
    rec = hits / len(gt_future[g])
    return prec, rec, recs

Ks = [5, 10, 20]
for K in Ks:
    vals = [precision_recall_at_k(g, K=K) for g in groups]
    vals = [v for v in vals if v is not None]
    P = np.mean([v[0] for v in vals]) if vals else np.nan
    R = np.mean([v[1] for v in vals]) if vals else np.nan
    print(f"@{K}: Precision={P:.4f} Recall={R:.4f}")

@5: Precision=0.0361 Recall=0.0706
@10: Precision=0.0330 Recall=0.0878
@20: Precision=0.0271 Recall=0.0984
